**⚠️ Technical Note:** This notebook was developed in a Google Colab environment. The raw bioclimatic and GEDI/VHM datasets are hosted in a private Google Drive directory. To replicate this study, users must provide their own raster assets or contact the author for access to the standardized 1km² Parquet files.

# 02: Climate Stressor Feature Engineering
**Project:** A Validated Predictive Framework for Climate-Smart Reforestation in Armenia

**Author:** Narek Ohanyan

In [ ]:
!pip install -q rioxarray geopandas xarray pandas numpy pyarrow fastparquet dask shap rasterio zarr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import pandas as pd
import xarray as xr
import rioxarray
import numpy as np
import gc
import geopandas as gpd
import warnings
from rasterio.enums import Resampling

In [ ]:
BASE_DIR = '/content/drive/MyDrive/FORACCA_Output_1_2'

In [ ]:

CLIPPED_DIR = f'{BASE_DIR}/data_raw/CHELSA_Clipped'

print("Scanning the clipped CHELSA directory...")
all_files = [f for f in os.listdir(CLIPPED_DIR) if f.endswith('.tif')]

# Parse filenames to organize the dataset
file_records = []
for f in all_files:
    # Pattern matches: Armenia_{var}_{month}_{year}.tif
    match = re.match(r'Armenia_(pr|tasmax|vpd)_(\d{2})_(\d{4})\.tif', f)
    if match:
        var, month, year = match.groups()
        file_records.append({
            'var': var,
            'date': pd.to_datetime(f'{year}-{month}-01'),
            'path': os.path.join(CLIPPED_DIR, f)
        })

df_files = pd.DataFrame(file_records)
print(f"Found {len(df_files)} valid monthly climate files.")

# Helper function to load a variable's time series
def load_variable_timeseries(var_name):
    print(f"Assembling time series for {var_name.upper()}...")
    df_var = df_files[df_files['var'] == var_name].sort_values('date')

    # Lazily open all files for this variable and concatenate along a new 'time' dimension
    datasets = [rioxarray.open_rasterio(row['path'], chunks={'x': 250, 'y': 250}).squeeze(drop=True) for _, row in df_var.iterrows()]
    time_index = xr.Variable('time', df_var['date'].values)

    # Concat and assign the time coordinate
    da = xr.concat(datasets, dim=time_index)
    da.name = var_name
    return da

# Build the master multidimensional dataset
ds = xr.Dataset({
    'pr': load_variable_timeseries('pr'),
    'tasmax': load_variable_timeseries('tasmax'),
    'vpd': load_variable_timeseries('vpd')
})

# --- CRITICAL UNIT CONVERSIONS ---
print("Applying physical unit conversions...")
# CHELSA V2.1 tasmax is stored as (Kelvin * 10). We convert directly to Celsius.
ds['tasmax'] = (ds['tasmax'] * 0.1) - 273.15
# CHELSA V2.1 VPD is stored in Pascals. Convert to kilopascals (kPa) for the FVS formula.
ds['vpd'] = ds['vpd'] / 1000.0
# pr (Precipitation) is in kg m-2 month-1, which is exactly equivalent to mm/month. No conversion needed.

print("\n✅ Multidimensional Climate Cube Assembled!")
ds

Scanning the clipped CHELSA directory...
Found 1514 valid monthly climate files.
Assembling time series for PR...


/tmp/ipykernel_3286/1021406136.py:40: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  da = xr.concat(datasets, dim=time_index)


Assembling time series for TASMAX...


/tmp/ipykernel_3286/1021406136.py:40: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  da = xr.concat(datasets, dim=time_index)


Assembling time series for VPD...
Applying physical unit conversions...

✅ Multidimensional Climate Cube Assembled!


<xarray.Dataset> Size: 2GB
Dimensions:      (x: 382, y: 885, time: 516)
Coordinates:
  * x            (x) float64 3kB 43.45 43.46 43.47 43.48 ... 46.61 46.62 46.63
  * y            (y) float64 7kB 38.85 38.85 38.85 38.85 ... 41.3 41.3 41.3
  * time         (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2021-12-01
    spatial_ref  int64 8B 0
Data variables:
    pr           (time, y, x) float32 698MB dask.array<chunksize=(1, 147, 250), meta=np.ndarray>
    tasmax       (time, y, x) float32 698MB dask.array<chunksize=(1, 147, 250), meta=np.ndarray>
    vpd          (time, y, x) float32 698MB dask.array<chunksize=(1, 147, 250), meta=np.ndarray>

In [ ]:
# Define the save path for the raw multidimensional cube
OUT_CUBE_NC = f'{BASE_DIR}/data_processed/Armenia_Climate_Cube_1979_2022.nc'

print(f"Compressing and saving the Raw Climate Cube to {OUT_CUBE_NC}...")
print("Please wait. This may take 5-10 minutes as it processes all 43 years of data...")

# Export the initial xarray dataset as a single NetCDF file
ds.to_netcdf(OUT_CUBE_NC)

print("✅ Raw Climate Cube saved successfully!")

Compressing and saving the Raw Climate Cube to /content/drive/MyDrive/FORACCA_Output_1_2/data_processed/Armenia_Climate_Cube_1979_2022.nc...
Please wait. This may take 5-10 minutes as it processes all 43 years of data...
✅ Raw Climate Cube saved successfully!


In [ ]:

print("--- 1. Calculating ANNUAL Deltas ---")
# Calculate Baseline (1979-2000) for the whole year
ds_annual_base = ds.sel(time=slice('1979-01-01', '2000-12-31')).mean(dim='time')

# Calculate Validation Epoch (2000-2018) for the whole year
ds_annual_recent = ds.sel(time=slice('2000-01-01', '2018-12-31')).mean(dim='time')

# Compute Annual Anomalies
ds_annual_delta = ds_annual_recent - ds_annual_base
ds_annual_delta = ds_annual_delta.rename({
    'vpd': 'Delta_VPD_Annual',
    'tasmax': 'Delta_Tmax_Annual',
    'pr': 'Delta_P_Annual'
})


print("--- 2. Calculating GROWING SEASON (GS) Deltas ---")
# Isolate May through September
ds_gs = ds.sel(time=ds['time.month'].isin([5, 6, 7, 8, 9]))

# Calculate Baseline (1979-2000) for GS only
ds_gs_base = ds_gs.sel(time=slice('1979-01-01', '2000-12-31')).mean(dim='time')

# Calculate Validation Epoch (2000-2018) for GS only
ds_gs_recent = ds_gs.sel(time=slice('2000-01-01', '2018-12-31')).mean(dim='time')

# Compute GS Anomalies
ds_gs_delta = ds_gs_recent - ds_gs_base
ds_gs_delta = ds_gs_delta.rename({
    'vpd': 'Delta_VPD_GS',
    'tasmax': 'Delta_Tmax_GS',
    'pr': 'Delta_P_GS'
})


print("--- 3. Merging into Master Delta Dataset ---")
# Combine all 6 anomaly layers into a single flat spatial map
ds_master_deltas = xr.merge([ds_annual_delta, ds_gs_delta])

print("\n✅ Annual and GS Deltas successfully calculated and merged!")
ds_master_deltas

--- 1. Calculating ANNUAL Deltas ---
--- 2. Calculating GROWING SEASON (GS) Deltas ---
--- 3. Merging into Master Delta Dataset ---

✅ Annual and GS Deltas successfully calculated and merged!


<xarray.Dataset> Size: 8MB
Dimensions:            (x: 382, y: 885)
Coordinates:
  * x                  (x) float64 3kB 43.45 43.46 43.47 ... 46.61 46.62 46.63
  * y                  (y) float64 7kB 38.85 38.85 38.85 ... 41.3 41.3 41.3
    spatial_ref        int64 8B 0
Data variables:
    Delta_P_Annual     (y, x) float32 1MB dask.array<chunksize=(147, 250), meta=np.ndarray>
    Delta_Tmax_Annual  (y, x) float32 1MB dask.array<chunksize=(147, 250), meta=np.ndarray>
    Delta_VPD_Annual   (y, x) float32 1MB dask.array<chunksize=(147, 250), meta=np.ndarray>
    Delta_P_GS         (y, x) float32 1MB dask.array<chunksize=(147, 250), meta=np.ndarray>
    Delta_Tmax_GS      (y, x) float32 1MB dask.array<chunksize=(147, 250), meta=np.ndarray>
    Delta_VPD_GS       (y, x) float32 1MB dask.array<chunksize=(147, 250), meta=np.ndarray>

In [ ]:
# Define the save path in the processed folder
OUT_DELTAS_NC = f'{BASE_DIR}/data_processed/Armenia_Climate_Deltas_1km.nc'

print(f"Saving Master Climate Deltas to {OUT_DELTAS_NC}...")
# Export the xarray dataset as a NetCDF file
ds_master_deltas.to_netcdf(OUT_DELTAS_NC)

print("✅ Saved successfully!")

Saving Master Climate Deltas to /content/drive/MyDrive/FORACCA_Output_1_2/data_processed/Armenia_Climate_Deltas_1km.nc...
✅ Saved successfully!


In [ ]:

warnings.filterwarnings("ignore")

# --- Define Paths ---

SHAPEFILE_PATH = f'{BASE_DIR}/data_raw/arm_admin0.geojson'
DELTAS_NC_PATH = f'{BASE_DIR}/data_processed/Armenia_Climate_Deltas_1km.nc'
NORTH_OUT = f'{BASE_DIR}/data_raw/Hansen_North_1km.tif'

print("Loading Master Grid & Shapefile...")
# We just need this to copy its 1km grid
ds_master_deltas = xr.open_dataset(DELTAS_NC_PATH)
template_1km = ds_master_deltas['Delta_VPD_GS']
armenia_gdf = gpd.read_file(SHAPEFILE_PATH)

print("Streaming, Clipping, and Reprojecting NORTH Tile...")
url1 = "https://storage.googleapis.com/earthenginepartners-hansen/GFC-2022-v1.10/Hansen_GFC-2022-v1.10_lossyear_50N_040E.tif"
rds1 = rioxarray.open_rasterio(url1, chunks=True)

# 1. Clip at 30m
clip1 = rds1.rio.clip(armenia_gdf.geometry, armenia_gdf.crs, drop=False)

# 2. Immediately shrink to 1km (Resampling.max flags ANY destruction in that 1km box)
hansen_north_1km = clip1.rio.reproject_match(template_1km, resampling=Resampling.max)
hansen_north_1km.name = 'lossyear'

# 3. Save to Drive and Dump RAM
print(f"Saving to {NORTH_OUT}...")
hansen_north_1km.rio.to_raster(NORTH_OUT)

del rds1, clip1, hansen_north_1km
gc.collect()
print("✅ North Tile Complete and RAM Cleared!")

Loading Master Grid & Shapefile...
Streaming, Clipping, and Reprojecting NORTH Tile...
Saving to /content/drive/MyDrive/FORACCA_Output_1_2/data_raw/Hansen_North_1km.tif...
✅ North Tile Complete and RAM Cleared!


In [ ]:
SOUTH_OUT = f'{BASE_DIR}/data_raw/Hansen_South_1km.tif'

print("Streaming, Clipping, and Reprojecting SOUTH Tile...")
url2 = "https://storage.googleapis.com/earthenginepartners-hansen/GFC-2022-v1.10/Hansen_GFC-2022-v1.10_lossyear_40N_040E.tif"
rds2 = rioxarray.open_rasterio(url2, chunks=True)

# 1. Clip at 30m
clip2 = rds2.rio.clip(armenia_gdf.geometry, armenia_gdf.crs, drop=False)

# 2. Immediately shrink to 1km
hansen_south_1km = clip2.rio.reproject_match(template_1km, resampling=Resampling.max)
hansen_south_1km.name = 'lossyear'

# 3. Save to Drive and Dump RAM
print(f"Saving to {SOUTH_OUT}...")
hansen_south_1km.rio.to_raster(SOUTH_OUT)

del rds2, clip2, hansen_south_1km, template_1km, ds_master_deltas
gc.collect()
print("✅ South Tile Complete and RAM Cleared!")

Streaming, Clipping, and Reprojecting SOUTH Tile...
Saving to /content/drive/MyDrive/FORACCA_Output_1_2/data_raw/Hansen_South_1km.tif...
✅ South Tile Complete and RAM Cleared!


In [ ]:

# --- Define Paths ---

DELTAS_NC_PATH = f'{BASE_DIR}/data_processed/Armenia_Climate_Deltas_1km.nc'
VHM_NC_PATH = f'{BASE_DIR}/data_processed/Armenia_VHM_1km.nc'
SHAPEFILE_PATH = f'{BASE_DIR}/data_raw/arm_admin0.geojson'
OUT_PARQUET = f'{BASE_DIR}/data_processed/Armenia_ML_Training_Data.parquet'
NORTH_LOSS = f'{BASE_DIR}/data_raw/Hansen_North_1km.tif'
SOUTH_LOSS = f'{BASE_DIR}/data_raw/Hansen_South_1km.tif'

print("1. Loading Master Grids and Fixing Coordinate Drift...")
ds_master_deltas = xr.open_dataset(DELTAS_NC_PATH)
armenia_gdf = gpd.read_file(SHAPEFILE_PATH)

# Mathematically anchor the Y-axis drift
clean_vars = {var: ds_master_deltas[var].dropna(dim='y', how='all') for var in ds_master_deltas.data_vars if var != 'spatial_ref'}
anchor_y, anchor_x = clean_vars['Delta_VPD_GS'].y.values, clean_vars['Delta_VPD_GS'].x.values
aligned_vars = {var: da.assign_coords(y=anchor_y, x=anchor_x) for var, da in clean_vars.items()}
ds_deltas_clean = xr.Dataset(aligned_vars).rio.write_crs("EPSG:4326")

print("2. Merging Lossyear (Red Band) with Xarray-Native Combine...")
t1_loss = rioxarray.open_rasterio(NORTH_LOSS).squeeze(drop=True)
t2_loss = rioxarray.open_rasterio(SOUTH_LOSS).squeeze(drop=True)

# THE FIX: Mask the '255' NoData value to NaN so the tiles become "transparent"
t1_loss = t1_loss.where(t1_loss != 255)
t2_loss = t2_loss.where(t2_loss != 255)

# Combine using xarray logic (Syunik will now show through the North's transparency)
loss_combined = t1_loss.combine_first(t2_loss)
loss_1km = loss_combined.rio.reproject_match(ds_deltas_clean['Delta_VPD_GS'])

print("3. Streaming and Merging Treecover2000 (Green Band)...")
url_tc_north = "https://storage.googleapis.com/earthenginepartners-hansen/GFC-2022-v1.10/Hansen_GFC-2022-v1.10_treecover2000_50N_040E.tif"
url_tc_south = "https://storage.googleapis.com/earthenginepartners-hansen/GFC-2022-v1.10/Hansen_GFC-2022-v1.10_treecover2000_40N_040E.tif"

# Load, clip, and reproject
tc_n = rioxarray.open_rasterio(url_tc_north, chunks=True).rio.clip(armenia_gdf.geometry, armenia_gdf.crs, drop=False)
tc_s = rioxarray.open_rasterio(url_tc_south, chunks=True).rio.clip(armenia_gdf.geometry, armenia_gdf.crs, drop=False)

tc_n_1km = tc_n.rio.reproject_match(ds_deltas_clean['Delta_VPD_GS']).squeeze(drop=True)
tc_s_1km = tc_s.rio.reproject_match(ds_deltas_clean['Delta_VPD_GS']).squeeze(drop=True)

# THE FIX: Mask the 255 and combine
tc_n_1km = tc_n_1km.where(tc_n_1km <= 100)
tc_s_1km = tc_s_1km.where(tc_s_1km <= 100)
treecover_1km = tc_n_1km.combine_first(tc_s_1km)

print("4. Applying Biological Masks...")
# Standard UN FAO Forest definition: > 15% canopy cover
valid_treecover = (treecover_1km > 15)
# No forest loss between 2000-2017
valid_loss = (loss_1km == 0) | (loss_1km > 17)

true_forest_mask = xr.where(valid_treecover & valid_loss, 1, np.nan)
ds_deltas_masked = ds_deltas_clean * true_forest_mask

print("5. Aligning VHM Forest Structure and Flattening...")
ds_vhm = xr.open_dataset(VHM_NC_PATH).rio.write_crs("EPSG:4326").rio.reproject_match(ds_deltas_clean['Delta_VPD_GS'])

# Drop metadata to prevent merge clashes
ds_deltas_masked = ds_deltas_masked.drop_vars('spatial_ref', errors='ignore')
ds_vhm = ds_vhm.drop_vars('spatial_ref', errors='ignore')

# Final Flattening
ds_final = xr.merge([ds_deltas_masked, ds_vhm])
df_ml = ds_final.to_dataframe().reset_index()

# Final clean: Drop NaNs and ensure trees exist (height > 0)
df_ml_clean = df_ml.dropna().copy()
df_ml_true_forest = df_ml_clean[df_ml_clean['vhm_mean'] > 0].copy()

# Save the unified North+South dataset
df_ml_true_forest.to_parquet(OUT_PARQUET, index=False)

print(f"\n✅ SUCCESS! Syunik and Northern forests unified.")
print(f"🌲 FINAL STABLE FOREST PIXELS (TOTAL COUNTRY): {len(df_ml_true_forest)} 🌲")

1. Loading Master Grids and Fixing Coordinate Drift...
2. Merging Lossyear (Red Band) with Xarray-Native Combine...
3. Streaming and Merging Treecover2000 (Green Band)...


/usr/local/lib/python3.12/dist-packages/dask/array/chunk.py:288: RuntimeWarning: invalid value encountered in cast
  return x.astype(astype_dtype, **kwargs)
/usr/local/lib/python3.12/dist-packages/dask/array/chunk.py:288: RuntimeWarning: invalid value encountered in cast
  return x.astype(astype_dtype, **kwargs)


4. Applying Biological Masks...
5. Aligning VHM Forest Structure and Flattening...

✅ SUCCESS! Syunik and Northern forests unified.
🌲 FINAL STABLE FOREST PIXELS (TOTAL COUNTRY): 4338 🌲
